# Albanian NER: open LLMs, zero-shot and few-shot

Open-weight instruction models (no API key, free Colab T4) label the same gold test set as the fine-tuned encoders.
Each model runs in three prompt settings:

| Setting | Prompt |
|---|---|
| `zero_plain` | Only the four types (PER, ORG, LOC, MISC), one line each |
| `zero_guidelines` | The four types + the main rules of the annotation guidelines |
| `few_guidelines` | Guidelines + 8 worked examples: 7 sentences from **training** documents (lightly shortened) and 1 written to illustrate MISC; never from the test set |

Decoding is greedy (no sampling), so every run is deterministic. The model answers with a JSON list of entities; the
strings are matched back to the words of the sentence. Strings that do not occur in the sentence count as lost (reported).
Models are loaded in 4-bit (NF4) so that an 8B model fits on the T4.

**Before you start:** *Runtime -> Change runtime type -> T4 GPU*, then *Runtime -> Run all*.
When asked, allow access to Google Drive: results are saved there after every setting, so nothing is lost if Colab disconnects,
and a re-run skips the settings that are already done.

In [ ]:
!pip -q install seqeval bitsandbytes accelerate
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4 GPU")

## 1. Data and storage

* **Colab:** a *Choose Files* button appears: pick `test.jsonl` (from `CORPUS/bio/`), then allow Google Drive.
* **Kaggle:** first add `test.jsonl` with *+ Add Input -> Upload -> New Dataset*; results go to `/kaggle/working`.

In [ ]:
import os, json, glob
if os.path.exists("/kaggle/input"):
    found = glob.glob("/kaggle/input/**/test.jsonl", recursive=True)
    if not found:
        raise FileNotFoundError("Add test.jsonl first: + Add Input -> Upload -> New Dataset")
    DATA, OUT = found[0], "/kaggle/working/ner_llm_results"
else:
    DATA = "/content/data/test.jsonl"
    os.makedirs("/content/data", exist_ok=True)
    try:
        from google.colab import files, drive
        if not os.path.exists(DATA):
            for name, content in files.upload().items():
                open(os.path.join("/content/data", name), "wb").write(content)
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/ner_llm_results"
    except ImportError:
        OUT = "/content/ner_llm_results"
os.makedirs(f"{OUT}/predictions", exist_ok=True)
gold = [json.loads(l) for l in open(DATA, encoding="utf-8")]
print(len(gold), "gold sentences | results go to", OUT)

## 2. Settings

Each model x setting takes roughly 15-45 minutes on a T4 (few-shot is the slowest: its prompts are long, so it runs 2 sentences at a time). All 6 runs take about 2-3 hours;
if the free GPU time runs out, run the notebook again later: finished runs are skipped.

In [ ]:
MODELS = {
    "Qwen3-8B":         "Qwen/Qwen3-8B",
    "Qwen2.5-7B-Instr": "Qwen/Qwen2.5-7B-Instruct",
}
SETTINGS = ["zero_plain", "zero_guidelines", "few_guidelines"]
BATCH = {"zero_plain": 8, "zero_guidelines": 4, "few_guidelines": 2}
MAX_NEW_TOKENS = 400

## 3. Prompts

In [ ]:
TYPES = ["PER", "ORG", "LOC", "MISC"]

TYPE_LINES = """PER: names of people (first name, surname or both).
ORG: named organisations and institutions: ministries, agencies, parliaments, courts, parties, companies, international organisations.
LOC: named places: countries, cities, regions, rivers, mountains.
MISC: other named entities: events (summits, conferences), named funds, programmes and projects, named systems, portals and products, nationalities used as names."""

GUIDELINES = """Annotation rules:
1. Only proper names. Common nouns are never entities, even if they refer to an institution: "ministria", "bashkitë", "qeveria e vendit".
2. Capital letters decide. A phrase that starts with a lowercase word is not an entity; if a lowercase word is followed by a name, only the name is the entity: "qeverisë së Hungarisë" -> only "Hungarisë" (LOC).
3. Documents are NOT entities: laws, codes, strategies, plans, agreements, memoranda, decisions, regulations, charters, the constitution, budgets, reports ("Kodi Zgjedhor", "Strategjia Kombëtare ...", "VKM"). Names inside a document title are still entities: "Kartës së Kombeve të Bashkuara" -> only "Kombeve të Bashkuara" (ORG).
4. Positions and roles are NOT entities: "Ministri i Shtetit për Diasporën", "Kryeministri", "deputetja", "zëvendësministre".
5. A single generic institution word without its full name is NOT an entity ("Ministrisë", "Drejtoria", "Parlamenti", "Byroja"). Exception: "Kuvendi" and "Qeveria" (capitalised, meaning this parliament / this government) are ORG.
6. Keep the inflected form exactly as written, including case endings and linking articles: "Ministrisë së Drejtësisë", "Bashkimin Evropian".
7. An acronym with a case suffix is one entity including the suffix: "BERZH-it", "RTM-së", "KQZ-në" (ORG). "TVSH" (VAT) is not an entity.
8. Entities never overlap; take the longest name: "Ushtria Çlirimtare e Kosovës" is one ORG (do not also mark "Kosovës").
9. "në" + a place ends a name: "Komisioni ... për Peshkimin në Mesdhe" -> ORG "Komisioni ... për Peshkimin" and LOC "Mesdhe".
10. Coordinated names are separate entities: "Ministrisë së Arsimit dhe Ministrisë së Shëndetësisë" -> two ORG. But one ministry with a long name stays one entity: "Ministria e Arsimit, Sportit dhe Rinisë".
11. A country or state as a place is LOC ("Republikës së Kosovës", "Shqipëria"); its government or parliament is ORG ("Kuvendi i Republikës së Kosovës")."""

FORMAT = """Return ONLY a JSON array, in the order the entities appear, one object per mention (repeat a name if it is mentioned twice):
[{"text": "<exact words copied from the sentence>", "type": "PER|ORG|LOC|MISC"}]
Copy "text" character by character from the sentence. If there are no entities, return []."""

EXAMPLES = [
    ("3. Interpelancë me Ministrin e Turizmit, Kulturës dhe Sportit, z. Blendi Gonxhja, kërkuar nga deputetja Ina Zhupa.",
     [("Ministrin e Turizmit, Kulturës dhe Sportit", "ORG"), ("Blendi Gonxhja", "PER"), ("Ina Zhupa", "PER")]),
    ("Dispozitat e caktuara që lidhen me mekanizmat rregullatorë dhe institucionalë të Bashkimit Evropian do të fillojnë të zbatohen në datën e anëtarësimit të Republikës së Shqipërisë në Bashkimin Evropian.",
     [("Bashkimit Evropian", "ORG"), ("Republikës së Shqipërisë", "LOC"), ("Bashkimin Evropian", "ORG")]),
    ("~ Ministri i Shtetit për Diasporën, me shkresën Nr. 983, datë 15.02.2019, i është drejtuar Ministrisë së Drejtësisë.",
     [("Ministrisë së Drejtësisë", "ORG")]),
    ("Këto ndryshime në Kodin e Punës bëjnë të mundur që institucionet publike apo private ta kenë të detyrueshme këtë rregull.",
     []),
    ("Aktualisht, 36 nga 46 vendet anëtare të CEPT-it janë edhe anëtare tek ECO.",
     [("CEPT-it", "ORG"), ("ECO", "ORG")]),
    ("Kemi qenë këtu gjatë diskutimit për buxhetin, dhe kemi marrë një angazhim nga ministrja e Shëndetësisë, që situata do të zgjidhet.",
     []),
    ("Sot Shqipëria nuk është më thjesht një vend që përfiton nga siguria kolektive e aleatëve të saj, por një vend që kontribuon në sigurinë e Evropës dhe të Aleancës së Atlantikut të Veriut.",
     [("Shqipëria", "LOC"), ("Evropës", "LOC"), ("Aleancës së Atlantikut të Veriut", "ORG")]),
    ("Aplikimi për certifikatën bëhet përmes portalit e-Albania, sipas Programit Kombëtar të Bursave.",
     [("e-Albania", "MISC"), ("Programit Kombëtar të Bursave", "MISC")]),
]

def system_prompt(setting):
    s = "You are an expert annotator of named entities in Albanian administrative and parliamentary text.\n\n"
    s += "Entity types:\n" + TYPE_LINES + "\n\n"
    if setting != "zero_plain":
        s += GUIDELINES + "\n\n"
    return s + FORMAT

def messages(sentence, setting):
    msgs = [{"role": "system", "content": system_prompt(setting)}]
    if setting.startswith("few"):
        for text, ents in EXAMPLES:
            msgs.append({"role": "user", "content": f"Sentence: {text}"})
            msgs.append({"role": "assistant", "content": json.dumps([{"text": t, "type": y} for t, y in ents],
                                                                    ensure_ascii=False)})
    msgs.append({"role": "user", "content": f"Sentence: {sentence}"})
    return msgs

def detok(tokens):
    s = " ".join(tokens)
    s = re.sub(r" ([,.;:!?%)\]”»…])", r"\1", s)
    s = re.sub(r"([(\[“„«]) ", r"\1", s)
    return s

import re
print(system_prompt("zero_guidelines")[:600], "...")

## 4. Matching the answer back to the words

In [ ]:
TOKEN = re.compile(r"\w+(?:[-‑’'`]\w+)*|[^\w\s]")
ALIAS = {"PERSON": "PER", "ORGANIZATION": "ORG", "ORGANISATION": "ORG", "LOCATION": "LOC", "GPE": "LOC",
         "MISCELLANEOUS": "MISC", "OTHER": "MISC"}

def parse(answer):
    a = re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()
    a = re.sub(r"^```(?:json)?|```$", "", a.strip(), flags=re.M).strip()
    m = re.search(r"\[.*\]", a, flags=re.S)
    if not m:
        return [], a.strip() in ("", "[]")
    try:
        items = json.loads(m.group(0))
    except Exception:
        items = [{"text": t, "type": y} for t, y in re.findall(r'"text"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([A-Za-z]+)"', m.group(0))]
        if not items:
            return [], False
    out = []
    for it in items if isinstance(items, list) else []:
        if isinstance(it, dict) and it.get("text"):
            y = ALIAS.get(str(it.get("type", "")).upper(), str(it.get("type", "")).upper())
            if y in TYPES:
                out.append((str(it["text"]).strip(), y))
    return out, True

def to_bio(tokens, ents):
    tags, used, lost, cursor = ["O"] * len(tokens), set(), 0, 0
    low = [t.lower() for t in tokens]
    for text, typ in ents:
        et = TOKEN.findall(text)
        if not et:
            lost += 1; continue
        n, found = len(et), None
        for seq, toks in ((et, tokens), ([t.lower() for t in et], low)):
            starts = [i for i in range(len(toks) - n + 1) if toks[i:i + n] == seq and not used & set(range(i, i + n))]
            if starts:
                found = next((i for i in starts if i >= cursor), starts[0]); break
        if found is None:
            lost += 1; continue
        tags[found] = "B-" + typ
        for i in range(found + 1, found + n):
            tags[i] = "I-" + typ
        used |= set(range(found, found + n)); cursor = found + n
    return tags, lost

## 5. Run

In [ ]:
import gc, time, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from seqeval.metrics import classification_report

def keep(tags, types):
    out, prev = [], "O"
    for t in tags:
        if t == "O" or t[2:] not in types:
            out.append("O"); prev = "O"; continue
        p, y = t.split("-", 1)
        if p == "I" and prev not in ("B-" + y, "I-" + y):
            p = "B"
        out.append(f"{p}-{y}"); prev = out[-1]
    return out

def scores(g, p, types):
    rep = classification_report(g, p, output_dict=True, zero_division=0)
    s = {"micro_F1": rep["micro avg"]["f1-score"], "macro_F1": rep["macro avg"]["f1-score"],
         "P": rep["micro avg"]["precision"], "R": rep["micro avg"]["recall"]}
    s.update({f"F1_{t}": rep.get(t, {}).get("f1-score", 0.0) for t in types})
    return s

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=q, device_map={"": 0})
    model.eval()
    return tok, model

def generate(tok, model, convs):
    prompts = []
    for c in convs:
        try:
            prompts.append(tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True, enable_thinking=False))
        except TypeError:
            prompts.append(tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True))
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

def generate_safe(tok, model, convs):
    try:
        return generate(tok, model, convs)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        if len(convs) == 1:
            raise
        h = len(convs) // 2
        return generate_safe(tok, model, convs[:h]) + generate_safe(tok, model, convs[h:])

def evaluate(model_key, setting, answers):
    rows, res = [], []
    for r, a in zip(gold, answers):
        ents, ok = parse(a)
        tags, lost = to_bio(r["tokens"], ents)
        rows.append({**{k: r[k] for k in ("id", "country", "doc_type", "tokens")}, "gold": r["ner_tags"],
                     "pred": tags, "answer": a, "parsed_ok": ok, "lost": lost, "n_ents": len(ents)})
    base = {"model": model_key, "setting": setting}
    stats = {"parse_fail": sum(not x["parsed_ok"] for x in rows), "lost_strings": sum(x["lost"] for x in rows),
             "answered_strings": sum(x["n_ents"] for x in rows)}
    for types, name in ((TYPES, "all"), (TYPES[:3], "all (3 types)")):
        g = [keep(x["gold"], types) for x in rows]; p = [keep(x["pred"], types) for x in rows]
        res.append({**base, "subset": name, "n": len(rows), **stats, **scores(g, p, types)})
    for key in ("country", "doc_type"):
        for val in sorted({x[key] for x in rows}):
            sub = [x for x in rows if x[key] == val]
            res.append({**base, "subset": f"{key}={val}", "n": len(sub),
                        **scores([x["gold"] for x in sub], [x["pred"] for x in sub], TYPES)})
    with open(f"{OUT}/predictions/{model_key}__{setting}.jsonl", "w", encoding="utf-8") as fh:
        for x in rows:
            fh.write(json.dumps(x, ensure_ascii=False) + "\n")
    return res

csv_path = f"{OUT}/llm_results_all.csv"
results = pd.read_csv(csv_path).to_dict("records") if os.path.exists(csv_path) else []
done = {(r["model"], r["setting"]) for r in results}
for model_key, model_id in MODELS.items():
    todo = [s for s in SETTINGS if (model_key, s) not in done]
    if not todo:
        print(f"{model_key}: all settings already done"); continue
    tok, model = load(model_id)
    for setting in todo:
        t0 = time.time()
        print(f"\n=== {model_key} | {setting} ===")
        part = f"{OUT}/partial_{model_key}__{setting}.json"
        done_ans = json.load(open(part)) if os.path.exists(part) else {}
        order = sorted(range(len(gold)), key=lambda i: len(gold[i]["tokens"]))
        todo_idx = [i for i in order if str(i) not in done_ans]
        if done_ans:
            print(f"  resuming: {len(done_ans)} sentences already done")
        b = BATCH[setting] if isinstance(BATCH, dict) else BATCH
        for k in range(0, len(todo_idx), b):
            idx = todo_idx[k:k + b]
            outs = generate_safe(tok, model, [messages(detok(gold[i]["tokens"]), setting) for i in idx])
            done_ans.update({str(i): a for i, a in zip(idx, outs)})
            if (k // b) % 10 == 0 or k + b >= len(todo_idx):
                json.dump(done_ans, open(part, "w"), ensure_ascii=False)
                print(f"  {len(done_ans)}/{len(gold)} sentences, {(time.time() - t0) / 60:.1f} min")
            torch.cuda.empty_cache()
        answers = [done_ans[str(i)] for i in range(len(gold))]
        res = evaluate(model_key, setting, answers)
        os.remove(part)
        results += res
        pd.DataFrame(results).to_csv(csv_path, index=False)
        r = res[0]
        print(f"F1 (4 types) = {r['micro_F1']:.3f} | F1 (PER/ORG/LOC) = {res[1]['micro_F1']:.3f} | "
              f"unparsable answers: {r['parse_fail']} | strings not found in the sentence: {r['lost_strings']}"
              f"/{r['answered_strings']} | {(time.time() - t0) / 60:.1f} min")
    del model; gc.collect(); torch.cuda.empty_cache()

## 6. Results

In [ ]:
df = pd.read_csv(csv_path)
pd.set_option("display.max_columns", None, "display.width", 250)
cols = ["model", "setting", "subset", "micro_F1", "P", "R", "F1_PER", "F1_ORG", "F1_LOC", "F1_MISC",
        "parse_fail", "lost_strings"]
display(df[df.subset.isin(["all", "all (3 types)"])][cols].round(3))
display(df[~df.subset.isin(["all", "all (3 types)"])].pivot_table(index="subset", columns=["model", "setting"],
                                                                     values="micro_F1").round(3))
import shutil
zip_path = shutil.make_archive(os.path.join(os.path.dirname(OUT) if OUT.startswith("/kaggle") else "/content",
                                              "ner_llm_results"), "zip", OUT)
print("zip:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    pass

**Send back** `ner_llm_results.zip` (it is also in your Google Drive, folder `ner_llm_results`).